# SImulations

In this notebook we simulate experiments to prove the efficacy of the Softmin Energy minimization method over other established global optimization methods.
The Softmin energy is defined as 
$$
    J_\beta(\mathbf{x}) = \sum_{i=1}^n \frac{e^{-\beta f (x_i)}f(x_i)}{\sum_{j=1}^n e^{-\beta f(x_j)}}
$$
with $\beta>0$, $\mathbf{x} = (x_1,\dots,x_n)$.

## Imports and utility functions

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle as pkl

from tqdm import tqdm
from simulations import *
from potentials import *
from viz import plot_entry_time_comparison
import multiprocessing as mp

plt.rcParams['font.size'] = 15
#uncomment to double the fontsize in plots

N_CPU = mp.cpu_count()


## Exit time from the maximizing regime

### Plot

In [ ]:
noise = 0.05
beta = 2
mean_exit_times, theory_bounds, x = [], [], []
q1_exit_times, q3_exit_times, std_exit_times = [], [], []

critical_values= []
for delta in np.arange(0.25,4.25,0.125):
    kwargs = one_dimensional_double_well(delta)
    f_star = kwargs['f_vec']([kwargs['critical_points'][1]]) - kwargs['f_vec']([kwargs['critical_points'][0]])
    if delta.is_integer(): delta = int(delta)
    string = f"softmin_static_double_well_init0_{delta}_{noise}".replace('.','p')+".pkl"
    with open(f"results/{string}", "rb") as f:
        exit_times = pkl.load(f)
    if f_star > 1/beta and 100/(2*99*beta) > (f_star-1/beta)**2:
        if len(critical_values) == 0:
            critical_values.append(x[-1])
    elif f_star > 1/beta and 100/(2*99*beta) <= (f_star-1/beta)**2:
        if len(critical_values) <= 1:
            critical_values.append(x[-1])
    x.append(f_star)


    mean_exit_times.append(np.mean(exit_times))
    delta_v_eff = max(min(f_star,100/(2*99*beta)),(max(f_star-1/beta,0))**2)
    theory_bounds.append(np.exp(delta_v_eff/noise))

    std_exit_times.append(np.std(exit_times))
    q1_exit_times.append(np.percentile(exit_times, 25))
    q3_exit_times.append(np.percentile(exit_times, 75))

def get_asymmetric_errorbars(mean, q1, q3):
    lower = np.array(mean) - np.array(q1)
    upper = np.array(q3) - np.array(mean)
    return [np.maximum(lower, 0), np.maximum(upper, 0)]


fig, ax = plt.subplots(figsize=(10, 6))
err_exit_times = get_asymmetric_errorbars(mean_exit_times, q1_exit_times, q3_exit_times)
ax.plot(x, mean_exit_times, label='Empirical Times', color='blue')
ax.vlines(critical_values, ymin=1e-2, ymax=1e5, colors='red', linestyles='dashed', label='Critical Values')
ax.set_yscale('log')
ax.set_ylim(1e-2, 1e5)
ax.set_xlabel('Potential Height')
ax.set_ylabel('Log Exit Time')
ax.set_title('Mean Exit Times from Maximizing Regime')
ax.legend()


## 1 Dimensional Double Well Potential

### Plots

In [ ]:
x = []
for delta in range(1, 31):
    kwargs = one_dimensional_double_well(delta)
    x.append(kwargs['critical_points'][2] - kwargs['critical_points'][0])

series = [
    {'label': 'SoftMin', 'filename_template': 'softmin_double_well_{p}.pkl',
     'color': '#0072B2', 'marker': 'o'},
    {'label': 'SoftMin Decay', 'filename_template': 'softmin_decay_highermaxsteps_double_well_{p}.pkl',
     'color': '#F0E442', 'marker': 'd'},
    {'label': 'Simulated Annealing', 'filename_template': 'sa_double_well_{p}.pkl',
     'color': '#E69F00', 'marker': 's'},
    {'label': 'CBO', 'filename_template': 'cbo_double_well_{p}.pkl',
     'color': '#009E73', 'marker': '^', 'mask_below': 1e4},
    {'label': 'PSO', 'filename_template': 'pso_double_well_{p}.pkl',
     'color': '#D55E00', 'marker': 'D'},
]

fig, ax = plot_entry_time_comparison(
    x=x, results_dir='results', series=series, params=range(1, 31),
    xlabel='Potential Barrier Height', title='Mean Entry Time vs Potential', grid=True,
)
plt.show()


## 2 Dimensional Quadruple Well Potential

In [ ]:
x_ = []
for A in range(2, 31):
    kwargs = unbalanced_quadruple_well_setting(A, -1)
    array_values = np.sort(np.array([np.float_(kwargs['f_vec'](point)) for point in kwargs['critical_points']]))
    x_.append(array_values[4] - array_values[3])

series = [
    {'label': 'SoftMin', 'filename_template': 'softmin_highermaxsteps_quadruple_well_{p}.pkl',
     'color': '#0072B2', 'marker': 'o'},
    {'label': 'SoftMin Decay', 'filename_template': 'softmin_decay_highermaxsteps_quadruple_well_{p}.pkl',
     'color': '#F0E442', 'marker': 'd'},
    {'label': 'Simulated Annealing', 'filename_template': 'sa_highermaxsteps_quadruple_well_{p}.pkl',
     'color': '#E69F00', 'marker': 's'},
    # CBO/PSO results also exist (cbo_quadruple_well_{p}.pkl, pso_quadruple_well_{p}.pkl) but
    # are excluded from this figure, matching the original notebook.
]

fig, ax = plot_entry_time_comparison(
    x=x_, results_dir='results', series=series, params=range(2, 31),
    xlabel='Potential Barrier Height', title='Mean Entry Time vs Potential', grid=True,
)
plt.show()


## Hgh Dimensional Ackley Function

In [ ]:
x = list(range(10, 100, 10))

series = [
    {'label': 'SoftMin', 'filename_template': 'SoftMin_decay_ackley_function_dim{p}.pkl',
     'color': '#0072B2', 'marker': 'o', 'mask_below': 1e5},
    {'label': 'CBO', 'filename_template': 'CBO_ackley_function_dim{p}.pkl',
     'color': '#009E73', 'marker': '^', 'mask_below': 1e5},
    {'label': 'PSO', 'filename_template': 'PSO_ackley_function_dim{p}.pkl',
     'color': '#D55E00', 'marker': 'D', 'mask_below': 1e5},
    # SimulatedAnnealing_ackley_function_dim{p}.pkl also exists but is excluded from this
    # figure, matching the original notebook.
]

fig, ax = plot_entry_time_comparison(
    x=x, results_dir='results', series=series,
    xlabel='Dimension', title='Mean Entry Time vs Dimension for Ackley Function', grid=True,
)
plt.show()
